# Run LASSQD

This guide builds a 24-hydrogen chain, prepares localized active orbitals, and
runs **LASSQD**: sample-based quantum diagonalization (SQD) as the fragment solver
for localized active space self-consistent field (LASSCF). Everything needed to
construct the system is below. Sampling runs locally with `ffsim.qiskit.FfsimSampler`.
[Fulqrum](https://qiskit-community.github.io/fulqrum/) constructs each projected
fragment Hamiltonian, which SciPy diagonalizes for SQD.

We use two H₁₂ fragments, each with twelve electrons in twelve spatial orbitals.
Each fragment is a singlet with six alpha and six beta electrons. Each fragment
has a separate 24-qubit circuit, simulated in its fixed electron-number sector
with 853,776 state amplitudes. The sampler adds 10% global depolarizing noise:
each measurement is replaced with that probability by a uniformly random
bitstring, which can have the wrong electron counts.

The fragments are too large to diagonalize in full at every step, so SQD works
in subspaces of at most 277 × 277 determinants, a tenth of the fragment space.
We select those subspaces with **trim SQD**, which screens several batches of
samples and diagonalizes only the configurations that carry the most weight.
The small number of atoms also lets us compare against classical LASSCF with
exact fragment solvers.

Run the code cells in order in a fresh Python kernel. The calculation needs
about 8 GB of memory and took about six minutes on eight cores. Runtime and
numerical details depend on your machine and package versions. With the
settings below, a
checked run with Fulqrum 0.4.5 and qiskit-addon-sqd 0.14.0 converged in
four cycles to about **−12.96944 Hartree**; the classical LASSCF reference
is about **−12.98823 Hartree**.

## Imports and the hybrid workflow

Each hybrid cycle performs the following steps:

1. Build an effective Hamiltonian for each fragment at the current orbitals.
2. Prepare a **local unitary cluster Jastrow (LUCJ)** circuit per fragment,
   initialized by compressed double factorization of CCSD amplitudes. Submit
   the fragments as separate measured circuits in one sampler job.
3. Use each fragment's measured bitstrings to select determinant subspaces.
   SQD diagonalizes the fragment Hamiltonian in these subspaces and returns
   energies and reduced density matrices (RDMs). Trim SQD screens several
   subspaces and diagonalizes the highest-weight part of all of them together.
4. Use those RDMs for one LASSCF orbital step, then repeat with the updated orbitals.

`run_lassqd` coordinates these steps. We supply the `prepare_fragment` builder below,
`FfsimSampler`, and a custom Fulqrum batch solver following the SQD addon's
[Fulqrum integration guide](https://qiskit.github.io/qiskit-addon-sqd/guides/integrate_fulqrum.html).

In [1]:
import time
from functools import partial
from math import comb

import ffsim
import fulqrum as fq
import numpy as np
import scipy.sparse.linalg
from ffsim.qiskit import FfsimSampler
from fulqrum.convert.integrals import integrals_to_fq_fermionic_op
from pyscf import cc, gto, scf
from qiskit import QuantumCircuit
from qiskit_addon_sqd.fermion import SCIResult, SCIState
from qiskit_addon_sqd.trim import TrimPolicy

from lassqd import (
    LASSCF,
    CycleResult,
    Fragment,
    FragmentSQD,
    initial_state,
    run_lassqd,
)

## Build H₂₄ and run Hartree–Fock

The atoms lie on the x-axis. Each H₁₂ group has 1.0 Å nearest-neighbor spacing,
and the closest atoms belonging to different groups are 2.0 Å apart.
STO-6G supplies one spatial basis function per hydrogen, giving 24 orbitals
for 24 electrons. All 24 orbitals will be active, so there are no inactive
core or external virtual orbitals in this example.

PySCF's molecular `spin` is the alpha-minus-beta electron count, or twice the spin
projection. We use `spin=0` and a restricted Hartree–Fock (RHF) calculation to
generate the initial orbitals. The fragment spin assignments are defined
separately in the next section.

PySCF and mrh compare their allocations with `max_memory`, in MB, which
defaults to 4,000. The SQD subspaces below take several GB of the same
process, so we raise the limit to 16,000. Raise it further if PySCF reports a
`MemoryError` or falls back to disk on a machine with more memory.

In [2]:
spacing, gap = 1.0, 2.0
x_positions = [spacing * i for i in range(12)]
x_positions += [x_positions[-1] + gap + spacing * i for i in range(12)]
mol = gto.M(
    atom=[("H", (x, 0.0, 0.0)) for x in x_positions],
    unit="Angstrom",
    basis="sto-6g",
    charge=0,
    spin=0,
    max_memory=16_000,
    verbose=0,
)
mf = scf.RHF(mol).run()
assert mf.converged, "RHF did not converge; inspect the mean-field calculation."

print(f"Electrons: {mol.nelectron}; spatial orbitals: {mf.mo_coeff.shape[1]}")
print(f"RHF total energy: {mf.e_tot:.10f} Hartree")

Electrons: 24; spatial orbitals: 24
RHF total energy: -12.5882363874 Hartree


## Define the active fragments and localize the orbitals

Atom indices are **zero-based**. `ncas_sub` counts spatial orbitals per fragment;
`nelecas_sub` gives each fragment's `(alpha, beta)` electron counts.
`spin_sub` contains spin **multiplicities** $2S+1$, so both H₁₂ singlets use 1.

| Fragment | Atom indices | Active orbitals | Alpha electrons | Beta electrons | Multiplicity |
| --- | --- | --- | --- | --- | --- |
| 0 | 0–11 | 12 | 6 | 6 | 1 |
| 1 | 12–23 | 12 | 6 | 6 | 1 |

Both fragments target $S=0$ and have spin projection $M_S=0$. The product of
two singlet fragment states is also a total-spin singlet. The Fulqrum solver
below fixes the electron counts but does not impose a total-spin penalty.
We check the returned fragment states' $\langle S^2\rangle$ after the calculation.

`localize_init_guess` assigns active orbitals to the atom groups and returns
orthonormal orbitals ordered by fragment. Here the entire orbital space is active,
so no orbital sorting or external orbital files are needed. For a larger basis,
select the active orbitals before localization.

In [3]:
fragment_atoms = (list(range(12)), list(range(12, 24)))
ncas_sub = (12, 12)
nelecas_sub = ((6, 6), (6, 6))
spin_sub = (1, 1)

las = LASSCF(mf, ncas_sub, nelecas_sub, spin_sub=spin_sub)
mo_initial = las.localize_init_guess(fragment_atoms, mf.mo_coeff)

np.testing.assert_allclose(
    mo_initial.T @ mf.get_ovlp() @ mo_initial,
    np.eye(mo_initial.shape[1]),
    atol=1e-10,
    rtol=0,
)
print(
    f"Fragments: {las.nfrags}; active orbitals: {las.ncas}; core orbitals: {las.ncore}"
)
print(f"Qubits per fragment: {[2 * n for n in ncas_sub]}")

Fragments: 2; active orbitals: 24; core orbitals: 0
Qubits per fragment: [24, 24]


## Compute a classical LASSCF reference

Use a separate LAS object with the same fragment definition and a copy of the
same initial orbitals. Its default classical fragment solvers provide a convenient
reference without sampling. It takes a few minutes, since each exact fragment
solve spans all 853,776 determinants, and is used only to assess the result;
LASSQD starts from `mo_initial`, not the optimized reference.

This reference is exact **within each fragment**. The LAS product form still
restricts the molecular wave function, so it is not a full-molecule FCI reference.

In [4]:
las_reference = LASSCF(mf, ncas_sub, nelecas_sub, spin_sub=spin_sub)
las_reference.kernel(mo_initial.copy())
assert las_reference.converged, "The classical LASSCF reference did not converge."
e_reference = float(las_reference.e_tot)
print(f"Classical LASSCF total energy: {e_reference:.10f} Hartree")

Classical LASSCF total energy: -12.9882256574 Hartree


## Define the fragment circuit builder

`run_lassqd` requires a `circuit_builder(fragment)` callable and leaves the
ansatz and connectivity to the calculation. Here we choose one LUCJ layer,
initialized by compressed double factorization of CCSD amplitudes. We use
nearest-neighbor same-spin interactions and opposite-spin pairs `(0, 0)`,
`(4, 4)` and `(8, 8)` for each twelve-orbital fragment. These are example settings; change
`interaction_pairs`, `n_reps`, and the compression settings below as needed.
`optimize=True` minimizes the factorization error in the CCSD amplitudes.

The builder receives a `Fragment`. Its integrals `fragment.h1` and `fragment.h2`
(chemists' notation) are expressed in the fragment's sampling basis: its ROHF
orbitals, computed once per cycle and also used by `FragmentSQD` to interpret
the measured occupations. The builder returns an unmeasured circuit with
`2 * norb` qubits: alpha orbitals on `0..norb-1` and beta orbitals on
`norb..2*norb-1`. `fragment.rohf()` provides the fragment's ROHF solution in
that basis as a PySCF object, without solving ROHF again, to start CCSD. Its
`max_memory` is PySCF's default, so the builder gives CCSD the molecule's limit.

In [5]:
def prepare_fragment(fragment: Fragment) -> QuantumCircuit:
    """Prepare this example's CCSD-initialized LUCJ state in the sampling basis."""
    norb, nelec = fragment.norb, fragment.nelec
    ccsd = cc.CCSD(fragment.rohf())
    ccsd.max_memory = mol.max_memory
    ccsd.verbose = 0
    ccsd.kernel()

    # Ansatz choices for this calculation; change these to explore other circuits.
    interaction_pairs = (
        [(p, p + 1) for p in range(norb - 1)],  # alpha-alpha
        [(p, p) for p in range(0, norb, 4)],  # alpha-beta
        [(p, p + 1) for p in range(norb - 1)],  # beta-beta
    )
    ucj_op = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
        ccsd.t2,
        t1=ccsd.t1,
        n_reps=1,
        interaction_pairs=interaction_pairs,
        optimize=True,
    )
    circuit = QuantumCircuit(2 * norb)
    # Alpha orbitals precede beta orbitals, as required by FragmentSQD.
    circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), circuit.qubits)
    circuit.append(ffsim.qiskit.UCJOpSpinUnbalancedJW(ucj_op), circuit.qubits)
    return circuit

## Define the Fulqrum batch solver

`FragmentSQD` passes batches of alpha/beta determinant integers and the current
fragment integrals to `sci_solver`. Convert those integrals to a Fulqrum operator,
project onto each batch's product subspace, and solve for its lowest eigenpair.
The one-dimensional case is just the single diagonal element.

The fragment Hamiltonian and its ROHF orbital basis change during LASSQD. Build
`hsub` inside every solver call from the supplied `h1` and `h2`, then reuse it
across that call's batches by updating the subspace. No constant energy is added
here: `FragmentSQD` adds the fragment's `h0` separately.

Fulqrum's vector ordering is the transpose of SQD's alpha-by-beta amplitude
matrix. Transpose the reshaped eigenvector and make it C-contiguous for PySCF.
Return it as an `SCIState` inside an `SCIResult`, together with the energy and
orbital occupancies used by configuration recovery. `FragmentSQD` computes the
RDMs from this state and rotates them back to the LAS orbital basis.


In [6]:
def solve_sci_batch_fulqrum(
    ci_batches: list[tuple[np.ndarray, np.ndarray]],
    h1: np.ndarray,
    h2: np.ndarray,
    norb: int,
    nelec: tuple[int, int],
    tol: float = 1e-12,
) -> list[SCIResult]:
    """Diagonalize each SQD batch using the current fragment Hamiltonian."""
    fermionic_op = integrals_to_fq_fermionic_op(
        one_body_integrals=np.ascontiguousarray(h1),
        two_body_integrals=np.ascontiguousarray(h2),
    )
    hsub = fq.SubspaceHamiltonian(
        fermionic_op.extended_jw_transformation(),
        fq.Subspace([["0" * norb], ["0" * norb]]),
    )

    results = []
    for ints_a, ints_b in ci_batches:
        strs_a = [format(x, f"0{norb}b") for x in ints_a]
        strs_b = [format(x, f"0{norb}b") for x in ints_b]
        subspace = fq.Subspace([strs_a, strs_b])
        hsub.update_subspace(subspace)
        diagonal = hsub.diagonal_vector()
        if len(subspace) == 1:
            energy = diagonal[0].real
            vec = np.ones(1, dtype=hsub.dtype)
        else:
            projected_hamiltonian = hsub.to_csr_linearoperator_fast()
            v0 = np.zeros(len(subspace), dtype=hsub.dtype)
            v0[np.argmin(diagonal)] = 1.0
            eigs, vecs = scipy.sparse.linalg.eigsh(
                projected_hamiltonian, k=1, which="SA", tol=tol, v0=v0
            )
            energy, vec = eigs[0], vecs[:, 0]
        occupancies = subspace.get_orbital_occupancies(
            probs=np.abs(vec) ** 2, norb=norb
        )
        # Fulqrum and SQD order the alpha/beta product space oppositely.
        state = SCIState(
            amplitudes=np.ascontiguousarray(vec.reshape(len(strs_b), len(strs_a)).T),
            ci_strs_a=ints_a,
            ci_strs_b=ints_b,
            norb=norb,
            nelec=nelec,
        )
        results.append(
            SCIResult(energy=energy, sci_state=state, orbital_occupancies=occupancies)
        )
    return results

## Configure trim SQD and local circuit sampling

Create one `FragmentSQD` instance per fragment. The main controls are:

| Setting | Here | Meaning |
| --- | --- | --- |
| `max_iterations` | 2 | Maximum configuration-recovery rounds per fragment solve |
| `num_batches` | 5 | Disjoint batches of samples screened per round |
| `samples_per_batch` | 300 | Number of sampled bitstring rows per batch |
| `max_dim` | 277 | Maximum number of alpha (and beta) strings in any subspace |
| `trim_ratio` | 0.3 | Fraction of each batch's strings kept after screening |
| `carryover_ratio` | 1.0 | Fraction of the screened strings carried into later rounds and cycles |
| `shots` | 10,000 | Measurements per fragment per hybrid cycle |
| `global_depolarizing` | 0.1 | Probability of replacing a measurement with a uniformly random bitstring |
| `max_cycles` | 10 | Maximum number of sampling/orbital-update cycles |
| `conv_tol` | $10^{-6}$ Hartree | Threshold on the change in total energy between cycles |

`policy=TrimPolicy(...)` selects trim SQD from `qiskit_addon_sqd.trim`. Each
round divides one pool of samples into `num_batches` disjoint batches and
diagonalizes each of them. It keeps the `trim_ratio` of each batch's strings
that carry the most weight, merges them, and diagonalizes the merged subspace.
That merged result is the one the round reports, so a round costs
`num_batches + 1` diagonalizations. Its highest-weight strings are carried into
the next round and the next cycle. `max_dim` bounds every one of these
subspaces, here to 277 strings per spin sector, or about 77,000 determinants.

The screening is what makes the subspace count. Without a policy, `FragmentSQD`
diagonalizes each batch and keeps the lowest-energy one. In our tests at the
same `max_dim`, that converged 1.4–2 times further from the LASSCF reference
than trim SQD, across three random seeds, at a similar cost. Removing the cap
improves either policy, but the subspaces then grow toward the full space: an
uncapped run with the default policy needed about an hour and over 100 GB.

On the first round the solver postselects for the requested alpha and beta
electron counts; the second uses the reported result's occupations for
configuration recovery. Recovery can stop early when both the energy and
occupation tolerances are met. We use a callback to track the latest
subspace dimensions. `symmetrize_spin=True` merges the selected alpha and beta
determinant strings before forming each product subspace.
This treats the two spin sectors equally, but does not enforce a singlet.
The Fulqrum batch solver finds the lowest-energy state in each subspace
using SciPy's `eigsh` with tolerance `1e-12`.

`FfsimSampler` simulates the native ffsim gates in each fragment's fixed
alpha/beta electron-number sector. Use `groups=[[0], [1]]` in `run_lassqd` to
submit the two measured fragment circuits separately in one job. The sampler
infers each fragment's orbital and electron counts from its Hartree–Fock
preparation gate. No pass manager is needed: keep the native ffsim gates intact.

Set `global_depolarizing=0.1` to replace each measurement with probability 10%
by a uniformly random 24-bit string. This gives configurations with negligible
circuit probability a chance to appear. The noise also generates strings with
incorrect alpha or beta electron counts, which SQD handles through postselection
and configuration recovery. The added samples broaden the selected subspace;
SQD still computes its energies from the fragment Hamiltonian.

The sampler and each SQD solver get their own random stream, spawned from one
seed, so a change in the draws of one does not shift the others. Each solver
keeps its stream in `solver.state` and continues it from cycle to cycle.
Restarting the kernel and running all cells reproduces the calculation, but
different package versions can change results.

In [7]:
# Retain just the latest round's dimensions for each fragment: with trim SQD,
# that of its merged subspace.
last_dimensions = [[] for _ in range(las.nfrags)]


def record_dimensions(dimensions: list[int], results: list[SCIResult]) -> None:
    dimensions[:] = [result.sci_state.amplitudes.size for result in results]


sampler_rng, *solver_rngs = np.random.default_rng(12345).spawn(las.nfrags + 1)

solvers = [
    FragmentSQD(
        samples_per_batch=300,
        max_iterations=2,
        num_batches=5,
        max_dim=277,
        policy=TrimPolicy(trim_ratio=0.3, carryover_ratio=1.0),
        sci_solver=solve_sci_batch_fulqrum,
        symmetrize_spin=True,
        seed=solver_rngs[ifrag],
        callback=partial(record_dimensions, last_dimensions[ifrag]),
    )
    for ifrag in range(las.nfrags)
]

sampler = FfsimSampler(seed=sampler_rng, global_depolarizing=0.1)

shots = 10_000
max_cycles = 10
conv_tol = 1e-6

## Run the hybrid calculation

`run_lassqd` starts from a `LASState`: here `initial_state` pairs the initial
orbitals with mrh's initial-guess fragment RDMs. Each cycle it samples the
fragment circuits, solves each fragment with its SQD solver, and then lets mrh
take one orbital step with the fragment RDMs. We explicitly pass
`prepare_fragment`, which chooses one LUCJ layer. The callback receives each
cycle's `CycleResult` and prints a compact progress line; its cycle index starts
at zero.

The call updates `las` and advances each solver's `state`. To repeat this
calculation from the same starting point and random streams, restart the kernel
and run all cells. Re-executing only this cell continues from the solvers'
current states.

In [8]:
start_time = time.perf_counter()


def report_cycle(cycle: int, step: CycleResult) -> None:
    elapsed = time.perf_counter() - start_time
    print(
        f"Cycle {cycle + 1:2d}: E = {step.e_tot:.10f} Hartree"
        f"  ({elapsed:.1f} s elapsed)"
    )


result = run_lassqd(
    las,
    initial_state(las, mo_initial),
    solvers,
    sampler,
    circuit_builder=prepare_fragment,
    groups=[[0], [1]],
    shots=shots,
    max_cycles=max_cycles,
    conv_tol=conv_tol,
    callback=report_cycle,
)
print(f"Converged: {result.converged}; cycles: {len(result.e_hist)}")

Cycle  1: E = -12.9668718402 Hartree  (41.8 s elapsed)


Cycle  2: E = -12.9686416844 Hartree  (86.6 s elapsed)


Cycle  3: E = -12.9694372663 Hartree  (129.7 s elapsed)


Cycle  4: E = -12.9694372671 Hartree  (176.0 s elapsed)
Converged: True; cycles: 4


## Inspect energies and convergence

`result.e_tot` is the molecular total energy, including nuclear repulsion.
`result.e_hist` contains one such energy per hybrid cycle. These are the energies
to compare with classical LASSCF. Individual fragment SQD callback energies use
fragment Hamiltonians and are not molecular total energies.

A checked run gave LASSQD **−12.9694372671 Hartree** and classical LASSCF
**−12.9882256574 Hartree**, a difference of about 19 mHartree.
Treat these as illustrative values. A LASSQD state is a product of fragment
states in the LAS form, so its energy is variational: it cannot fall below the
lowest LAS energy, which classical LASSCF approximates with exact fragment
solvers. The difference measures mostly what the truncated subspaces miss. `result.converged` means only that the last absolute energy change is
smaller than `conv_tol`. Here that happens once carryover settles the subspaces,
so it does not certify accuracy relative to the reference.

Each fragment's fixed-electron determinant space has
$\binom{12}{6}\binom{12}{6}=853{,}776$ determinants. Compare the final SQD
subspace dimensions printed below with this full-space dimension and with the
$277^2 = 76{,}729$ allowed by `max_dim`. A larger `max_dim` or `trim_ratio`
lowers the energy at a higher cost in time and memory.

The current implementation uses only the alpha one-electron Hamiltonian
`h1s[0]` for both circuit construction and SQD, even when the embedding Hamiltonian
is spin-dependent. Agreement with classical LASSCF is therefore not a
general guarantee for spin-polarized fragments.

In [9]:
print("Cycle       Total energy (Hartree)       Delta E (Hartree)")
for cycle, energy in enumerate(result.e_hist, start=1):
    delta = "--" if cycle == 1 else f"{energy - result.e_hist[cycle - 2]:+.3e}"
    print(f"{cycle:5d}       {energy: .10f}             {delta:>10}")

energy_difference = float(result.e_tot - e_reference)
print(f"LASSQD total energy:           {result.e_tot:.10f} Hartree")
print(f"Classical LASSCF total energy: {e_reference:.10f} Hartree")
print(f"LASSQD minus reference:        {energy_difference:+.3e} Hartree")

assert np.isfinite(result.e_hist).all(), (
    "The energy history contains a nonfinite value."
)
assert result.converged, (
    "Increase max_cycles and inspect the energy changes before continuing."
)
assert abs(result.e_hist[-1] - result.e_hist[-2]) < conv_tol

for ifrag, solver in enumerate(solvers):
    norb = ncas_sub[ifrag]
    nalpha, nbeta = nelecas_sub[ifrag]
    full_dimension = comb(norb, nalpha) * comb(norb, nbeta)
    print(
        f"Fragment {ifrag}: final-round SQD subspace dimensions "
        f"{last_dimensions[ifrag]} (full space: {full_dimension})"
    )

Cycle       Total energy (Hartree)       Delta E (Hartree)
    1       -12.9668718402                     --
    2       -12.9686416844             -1.770e-03
    3       -12.9694372663             -7.956e-04
    4       -12.9694372671             -8.189e-10
LASSQD total energy:           -12.9694372671 Hartree
Classical LASSCF total energy: -12.9882256574 Hartree
LASSQD minus reference:        +1.879e-02 Hartree
Fragment 0: final-round SQD subspace dimensions [20164] (full space: 853776)
Fragment 1: final-round SQD subspace dimensions [22201] (full space: 853776)


## Check electron counts and the returned wave function

`result.state` is a `LASState`: its fragment RDMs `casdm1frs` and `casdm2fr` are
expressed in the active orbitals of its `mo_coeff`. Combining them with the
initial orbitals would describe a different wave function. `result.results`
holds the last cycle's fragment results, including each selected SQD state.

For fragment `i`, the spin-separated 1-RDM has shape
`(nroots, 2, n_i, n_i)` and the spin-summed 2-RDM has shape
`(nroots, n_i, n_i, n_i, n_i)`. This calculation has one root and twelve
orbitals per fragment. Tracing each spin block of root zero should recover
`(6, 6)` for both fragments. The H₁₂ ground states are singlets, but a
truncated subspace need not contain every determinant a singlet needs, so the
selected SQD states have a small $\langle S^2\rangle$ rather than zero; we
check that it stays below 0.05.

Finally, evaluate the energy from the returned orbitals and RDMs to check that
they are consistent with `result.e_tot`.

In [10]:
state = result.state
for ifrag, (dm1, dm2, nelec) in enumerate(
    zip(state.casdm1frs, state.casdm2fr, nelecas_sub)
):
    norb = ncas_sub[ifrag]
    assert dm1.shape == (1, 2, norb, norb)
    assert dm2.shape == (1, norb, norb, norb, norb)
    assert np.isfinite(dm1).all() and np.isfinite(dm2).all()
    spin_sq = result.results[ifrag].sci_state.spin_square()
    assert spin_sq < 0.05, f"fragment {ifrag}: <S^2> = {spin_sq:.3e}"
    populations = np.trace(dm1[0], axis1=-2, axis2=-1)
    np.testing.assert_allclose(populations, nelec, atol=1e-7, rtol=0)
    print(
        f"Fragment {ifrag}: 1-RDM {dm1.shape}, 2-RDM {dm2.shape}; "
        f"N_alpha = {populations[0]:.6f}, N_beta = {populations[1]:.6f}; "
        f"<S^2> = {spin_sq:.3e}"
    )

energy_from_rdms = las.energy_nuc() + las.energy_elec(
    mo_coeff=state.mo_coeff,
    casdm1frs=list(state.casdm1frs),
    casdm2fr=list(state.casdm2fr),
)
np.testing.assert_allclose(energy_from_rdms, result.e_tot, atol=1e-8, rtol=0)
print(f"Energy reconstructed from orbitals and RDMs: {energy_from_rdms:.10f} Hartree")

Fragment 0: 1-RDM (1, 2, 12, 12), 2-RDM (1, 12, 12, 12, 12); N_alpha = 6.000000, N_beta = 6.000000; <S^2> = 5.086e-03
Fragment 1: 1-RDM (1, 2, 12, 12), 2-RDM (1, 12, 12, 12, 12); N_alpha = 6.000000, N_beta = 6.000000; <S^2> = 4.786e-03
Energy reconstructed from orbitals and RDMs: -12.9694372671 Hartree


## Adapt the example

- **Geometry and basis:** Change the coordinates in Å and the PySCF basis, then
  recompute RHF and localize new initial orbitals. A larger basis introduces
  additional orbitals; choose an active space rather than assuming all orbitals
  belong to these two twelve-orbital fragments.
- **Active spaces and spins:** Update the zero-based atom groups, `ncas_sub`,
  `(alpha, beta)` electron counts, and multiplicities together. Account for
  doubly occupied inactive core orbitals in the molecular electron count. Each
  fragment needs two qubits per active spatial orbital. For a selected active
  subset, `las.sort_mo` uses **one-based orbital indices by default**; place the
  selected orbitals in the active block before calling `localize_init_guess`.
  Disable `symmetrize_spin` for unequal alpha and beta electron counts.
  For other target spins, adapt the spin checks and use an eigensolver with
  an appropriate spin constraint if the desired state is not the ground state.
- **Circuit ansatz:** Edit `prepare_fragment` to choose interaction pairs,
  `n_reps`, and compression settings, or supply another builder via `circuit_builder`.
  Keep the fragment ROHF orbital basis and alpha/beta qubit ordering so SQD
  interprets the measured occupations correctly.
- **Sampling and SQD cost:** More shots improve the chance of observing rare
  configurations. `max_dim` bounds the cost of every diagonalization, and
  `trim_ratio` how much of each screened batch reaches the merged subspace.
  Each batch adds one diagonalization per round, and recovery iterations repeat
  them all. These controls increase cost in different ways; inspect the energy
  history and SQD subspace dimensions as you change them. The default
  `TrimPolicy()` keeps a tenth of each batch and carries over a tenth of the
  merged strings, which is too little for these fragments.
- **Memory:** Raise `max_memory` in `gto.M` along with `max_dim`: the SQD
  subspaces live in the same process as PySCF and mrh.
- **Convergence:** If the cycle cap is reached while energies are still changing,
  increase `max_cycles`. If the energy fluctuates because of finite sampling,
  increase sampling or batch coverage and assess the fluctuations before choosing
  `conv_tol`. Tighter energy thresholds alone do not guarantee a more accurate
  wave function.

After making changes, restart the kernel and run all cells to rebuild the
molecule, reference, fragment solvers, and sampler consistently. The larger
[FeFe examples](https://github.com/joannaqw/lassqd_code/tree/main/examples) illustrate other systems and execution
settings once this local workflow is familiar.